## Setup Notebook

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import glob
import os
from pathlib import Path
from preprocessing import WindFarmProcessor

## Setup Paths

In [ ]:
# Adjust these to match your local project directory structure
RAW_DATA_PATH = Path("./data/raw/zenodo_windfarm_data")
PROCESSED_PATH = Path("./data/processed")
CONFIG_PATH = Path("./config/feature_map.yaml")

# Create output directory if it doesn't exist
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

## Initialize Processor and Run Batch Processing

In [ ]:
processor = WindFarmProcessor(config_path=CONFIG_PATH, window_days=30)

In [ ]:
# This loop handles 95 turbines without crashing your RAM
print("Starting Batch Processing...")
processor.process_all_turbines(raw_data_root=RAW_DATA_PATH, output_dir=PROCESSED_PATH)
print("Batch Processing Complete!")

## Load Final Harmonized Dataset

In [ ]:
def load_final_dataset(processed_dir):
    """
    Reads all processed Parquet files and performs final fleet-wide cleanup.
    """
    parquet_files = list(Path(processed_dir).glob("*.parquet"))
    
    if not parquet_files:
        raise FileNotFoundError("No processed parquet files found!")

    print(f"Loading {len(parquet_files)} turbines into memory...")
    
    # Concatenate all turbine dataframes
    # This is much smaller than the raw CSVs because we dropped ~90% of columns
    full_df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)
    
    # Sort globally by farm, asset, and time
    full_df = full_df.sort_values(['farm_id', 'asset_id', 'time_stamp'])
    
    # 5. FINAL FLEET-WIDE IMPUTATION
    # We fill Site B's missing hydraulic_temp with the average of Site A and C
    # We use the method from our processor class
    full_df = processor.impute_missing_sensors(full_df)
    
    return full_df

# Execute the load
df_final = load_final_dataset(PROCESSED_PATH)

## Data Verification

In [ ]:
print("\n--- Final Dataset Summary ---")
print(f"Total Rows: {len(df_final):,}")
print(f"Total Failures (Target=1): {df_final['target'].sum():,}")
print(f"Memory Usage: {df_final.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Check Site B's hydraulic indicators
site_b_check = df_final[df_final['farm_id'] == 'B'].head(1)
print(f"\nSite B 'has_hydraulic_temp' flag: {site_b_check['has_hydraulic_temp'].values[0]}")

# Display the head of the clean modeling data
df_final.head()

## Sandbox